In [ ]:
# @title 1.1 🔍 Check GPU
import torch

print("🔍 GPU Check:")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        print(f"   GPU {i}: {props.name} ({mem_gb:.1f} GB)")
else:
    print("❌ GPU not found! Enable GPU in Kaggle Settings.")

In [ ]:
# @title 1.2 📦 Clone Repository & Install Dependencies
import os
import sys

REPO_URL = "https://github.com/ngnam1104/TriMedAgent.git"
WORK_DIR = "/kaggle/working/TriMedAgent"

if not os.path.exists(WORK_DIR):
    print("📥 Cloning TriMedAgent...")
    !git clone {REPO_URL} {WORK_DIR}
else:
    print("✅ Repository exists.")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# Install dependencies
print("\n📦 Installing training dependencies...")
!pip install -q unsloth
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q datasets wandb huggingface_hub

print("\n✅ Installation Complete!")

In [ ]:
# @title 1.3 🔑 Setup HuggingFace Token
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Get token from Kaggle secrets
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace!")
except Exception as e:
    print("⚠️ Could not get HF_TOKEN from secrets.")
    print("   Add HF_TOKEN to Kaggle Secrets, or login manually:")
    HF_TOKEN = input("Enter HuggingFace Token: ")
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("✅ Logged in!")

---
## 2️⃣ 📊 Prepare Training Data

In [ ]:
# @title 2.1 📥 Download or Create Training Dataset
import os
import json
from pathlib import Path

DATA_DIR = Path("data/sft_dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Option 1: Download from HuggingFace Dataset
# !huggingface-cli download ngnam1104/trimedagent-sft-data --local-dir data/sft_dataset

# Option 2: Use sample data for demo
SAMPLE_DATA = [
    {
        "image": None,
        "conversations": [
            {"role": "user", "content": "<image>\nTìm các nốt mờ bất thường trong phổi."},
            {"role": "assistant", "content": '{"thought": "Cần detect các vùng nodule trong phổi", "action": "GroundingDINO", "action_input": {"prompt": "lung nodule"}}'}
        ]
    },
    {
        "image": None,
        "conversations": [
            {"role": "user", "content": "<image>\nPhân tích ảnh X-quang ngực này."},
            {"role": "assistant", "content": '{"thought": "Cần phân tích toàn bộ ảnh X-quang", "action": "LLaVA", "action_input": {"question": "Describe any abnormalities in this chest X-ray"}}'}
        ]
    },
    {
        "image": None,
        "conversations": [
            {"role": "user", "content": "<image>\nTìm vùng tim to (cardiomegaly) nếu có."},
            {"role": "assistant", "content": '{"thought": "Cần detect cardiomegaly - tim to", "action": "GroundingDINO", "action_input": {"prompt": "enlarged heart cardiomegaly"}}'}
        ]
    },
    {
        "image": None,
        "conversations": [
            {"role": "user", "content": "<image>\nSegment vùng đã được detect."},
            {"role": "assistant", "content": '{"thought": "Cần segment chi tiết các vùng detected", "action": "MedSAM", "action_input": {"boxes": "[[100, 100, 200, 200]]"}}'}
        ]
    },
    {
        "image": None,
        "conversations": [
            {"role": "user", "content": "<image>\nCó dấu hiệu viêm phổi không?"},
            {"role": "assistant", "content": '{"thought": "Kiểm tra dấu hiệu pneumonia", "action": "GroundingDINO", "action_input": {"prompt": "pneumonia consolidation infiltrate"}}'}
        ]
    }
]

# Expand sample data
expanded_data = SAMPLE_DATA * 20  # 100 samples for demo

# Save as JSONL
output_file = DATA_DIR / "train.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for item in expanded_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"✅ Created {len(expanded_data)} training samples")
print(f"📁 Saved to: {output_file}")

---
## 3️⃣ 🚀 Training Configuration

In [ ]:
# @title 3.1 ⚙️ Training Config
from dataclasses import dataclass
from typing import List

@dataclass
class SFTTrainingConfig:
    # Model
    base_model: str = "unsloth/llava-v1.5-7b-hf-bnb-4bit"
    
    # LoRA
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    target_modules: List[str] = None
    
    # Training
    num_epochs: int = 3
    batch_size: int = 2
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    max_seq_length: int = 2048
    
    # Output
    output_dir: str = "checkpoints/sft_adapter"
    hub_model_id: str = "ngnam1104/trimedagent-sft-v1"
    
    def __post_init__(self):
        if self.target_modules is None:
            self.target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"]

config = SFTTrainingConfig()
print("✅ Config ready!")
print(f"   Base: {config.base_model}")
print(f"   LoRA: r={config.lora_r}, α={config.lora_alpha}")
print(f"   Epochs: {config.num_epochs}")

In [ ]:
# @title 3.2 📚 Load Model with Unsloth
from unsloth import FastLanguageModel

print(f"🚀 Loading {config.base_model}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.base_model,
    max_seq_length=config.max_seq_length,
    dtype=None,  # Auto detect
    load_in_4bit=True,
)

print("✅ Model loaded!")

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=config.lora_r,
    target_modules=config.target_modules,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ LoRA adapters added!")

In [ ]:
# @title 3.3 📊 Prepare Dataset
from torch.utils.data import Dataset
import json

SYSTEM_PROMPT = """You are TriMed-Agent, a medical AI assistant.
Analyze medical images and respond with structured JSON plans.
Format:
{
    "thought": "your reasoning",
    "action": "tool name (GroundingDINO, MedSAM, RAG, or answer)",
    "action_input": {"param": "value"}
}"""

class SFTDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=2048):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []
        
        with open(data_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    self.data.append(json.loads(line))
                    
        print(f"Loaded {len(self.data)} examples")
    
    def __len__(self):
        return len(self.data)
    
    def _format_conversation(self, item):
        convs = item.get('conversations', [])
        parts = [f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"]
        
        for turn in convs:
            if turn['role'] == 'user':
                parts.append(f"{turn['content']} [/INST] ")
            else:
                parts.append(f"{turn['content']} </s>")
        
        return "".join(parts)
    
    def __getitem__(self, idx):
        text = self._format_conversation(self.data[idx])
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': encodings['input_ids'].squeeze().clone()
        }

train_dataset = SFTDataset(
    "data/sft_dataset/train.jsonl",
    tokenizer,
    config.max_seq_length
)

print(f"✅ Dataset ready: {len(train_dataset)} samples")

---
## 4️⃣ 🔥 Start Training

In [ ]:
# @title 4.1 🚂 Run Training
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Training arguments
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=3,
    report_to="none",  # Set to "wandb" if using W&B
    optim="paged_adamw_8bit",
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("🚂 Starting SFT Training...")
print(f"   Samples: {len(train_dataset)}")
print(f"   Epochs: {config.num_epochs}")
print(f"   Batch: {config.batch_size} x {config.gradient_accumulation_steps}")
print("="*50)

trainer.train()

print("\n✅ Training Complete!")

In [ ]:
# @title 4.2 💾 Save Adapter Locally
from pathlib import Path
import json

output_path = Path(config.output_dir) / "final"
output_path.mkdir(parents=True, exist_ok=True)

# Save model
trainer.save_model(str(output_path))
tokenizer.save_pretrained(str(output_path))

# Save training config
config_dict = {
    'base_model': config.base_model,
    'lora_r': config.lora_r,
    'lora_alpha': config.lora_alpha,
    'target_modules': config.target_modules,
    'num_epochs': config.num_epochs,
}
with open(output_path / "training_config.json", 'w') as f:
    json.dump(config_dict, f, indent=2)

print(f"✅ Adapter saved to: {output_path}")

---
## 5️⃣ 🚀 Push to HuggingFace

In [ ]:
# @title 5.1 📤 Push Adapter to HuggingFace Hub
from huggingface_hub import HfApi

REPO_ID = config.hub_model_id  # e.g., "ngnam1104/trimedagent-sft-v1"

print(f"📤 Pushing to HuggingFace: {REPO_ID}")

try:
    # Push model
    model.push_to_hub(REPO_ID, use_auth_token=True)
    tokenizer.push_to_hub(REPO_ID, use_auth_token=True)
    
    print(f"\n✅ Successfully pushed to: https://huggingface.co/{REPO_ID}")
    
except Exception as e:
    print(f"❌ Push failed: {e}")
    print("\nTry manual upload:")
    print(f"   huggingface-cli upload {REPO_ID} {output_path}")

In [ ]:
# @title 5.2 📋 Create Model Card
model_card = f"""---
license: apache-2.0
tags:
  - medical
  - vision
  - llava
  - lora
  - trimedagent
base_model: {config.base_model}
datasets:
  - custom
language:
  - en
  - vi
---

# TriMedAgent SFT Adapter

LoRA adapter for TriMedAgent - Medical Visual Agent.

## Training Details

- **Base Model**: `{config.base_model}`
- **LoRA Rank**: {config.lora_r}
- **LoRA Alpha**: {config.lora_alpha}
- **Target Modules**: {config.target_modules}
- **Epochs**: {config.num_epochs}

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained("{config.base_model}")
model = PeftModel.from_pretrained(base_model, "{REPO_ID}")
```

## Citation

```
@misc{{trimedagent,
  title={{TriMedAgent: Medical Visual Agent}},
  author={{Team}},
  year={{2025}}
}}
```
"""

# Save model card
with open(output_path / "README.md", 'w') as f:
    f.write(model_card)

print("✅ Model card created!")
print(f"\n📋 Next step: Run GRPO/RL training with this SFT adapter")
print(f"   Adapter path: {REPO_ID}")

---
## 🎉 Done!

SFT Training hoàn tất. Adapter đã được push lên HuggingFace.

**Next Steps:**
1. Chạy notebook `02_rl_grpo_training.ipynb` để fine-tune với GRPO
2. Hoặc test adapter ngay với notebook `03_demo.ipynb`